# 03 · Memoria e contesto

Di base ogni chiamata al modello è **senza memoria**. Qui vediamo come dare all'agente:
1. **memoria di breve termine** dentro una conversazione (checkpoint + `thread_id`);
2. **memoria di lungo termine** che sopravvive tra conversazioni (uno store + tool).

## Obiettivi, prerequisiti e modalità di lettura

Confronterai assenza di memoria, checkpoint per thread e memoria esterna. Durata: 25–35 minuti. Osserva sempre quale confine conserva ciascun dato.

Ogni blocco di codice è preceduto da una spiegazione e seguito da un **output
atteso**. Quando interviene un modello, l'output atteso descrive proprietà e
invarianti, non una frase letterale. Esegui le celle in ordine e non saltare i
casi negativi: mostrano il confine del meccanismo, non un incidente del corso.

## Setup (autonomo)

Ogni notebook è **indipendente**: non importa nulla dal progetto. Qui carichiamo la chiave
API dal file `.env` e creiamo un modello. Esegui le celle in ordine dall'alto verso il basso.

### Spiegazione del blocco · Setup isolato

La configurazione viene caricata senza importare il package finale, mantenendo il notebook autonomo.

In [ ]:
# Carichiamo le variabili d'ambiente dal file `.env`.
# Lo cerchiamo nella cartella corrente e in quelle superiori, così il notebook
# funziona sia se avviato dalla radice del progetto sia dalla cartella `notebooks`.
import os
from pathlib import Path

from dotenv import load_dotenv


def trova_env() -> Path:
    for cartella in (Path.cwd(), *Path.cwd().resolve().parents):
        if (cartella / ".env").is_file():
            return cartella / ".env"
    raise FileNotFoundError("File .env non trovato: copia .env.example in .env e aggiungi la chiave.")


env_file = trova_env()
load_dotenv(env_file, override=False)          # carica le variabili senza sovrascrivere quelle già presenti
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY mancante nel file .env"
print("Ambiente caricato da:", env_file)

### Output atteso

Percorso `.env` utilizzato.

### Spiegazione del blocco · Client del modello

Lo stesso modello verrà usato prima senza memoria e poi dentro un graph con checkpointer, così il confronto è significativo.

In [ ]:
# `ChatOpenAI` è il wrapper LangChain attorno al modello.
# Lo creiamo una volta e lo riusiamo in tutto il notebook.
from langchain_openai import ChatOpenAI

MODELLO = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")   # modello economico, va bene per imparare
model = ChatOpenAI(
    model=MODELLO,
    use_responses_api=True,   # API "responses" di OpenAI
    store=False,              # non conservare la conversazione sui server OpenAI
)
print("Modello pronto:", MODELLO)

### Output atteso

Nome del modello configurato.

## 1 · Nessuna memoria (il problema)

Due chiamate separate non condividono nulla: la seconda non "ricorda" la prima.

### Spiegazione del blocco · Dimostrazione stateless

Le due chiamate sono indipendenti: la seconda non riceve il primo messaggio. Non è un difetto del modello, ma dell'input fornito.

In [ ]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content="Mi chiamo Nico.")])
seconda = model.invoke([HumanMessage(content="Come mi chiamo?")])
print(seconda.text)   # non può saperlo: le due chiamate sono scollegate

### Output atteso

Il modello dichiara di non conoscere il nome. La forma esatta varia.

## 2 · Memoria di breve termine con checkpoint

Un **checkpointer** salva lo stato della conversazione. Se usiamo lo stesso `thread_id`,
l'agente ritrova i messaggi precedenti e quindi "ricorda".

### Spiegazione del blocco · Agente con checkpointer

`InMemorySaver` conserva lo stato per chiave di configurazione. È adatto alla lezione; un processo reale usa storage persistente.

In [ ]:
# InMemorySaver tiene lo stato in RAM (per imparare). In produzione: SQLite/Postgres.
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

agente = create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver(),   # abilita la memoria per-thread
)

### Output atteso

Nessun output. `agente` espone memoria breve per thread.

### Spiegazione del blocco · Stesso thread, memoria condivisa

Le due invocazioni usano lo stesso `thread_id`. Il checkpointer ricostruisce la cronologia prima della seconda chiamata.

In [ ]:
# `thread_id` identifica la conversazione: stesso id = stessa memoria.
config = {"configurable": {"thread_id": "conversazione-1"}}

agente.invoke({"messages": [{"role": "user", "content": "Mi chiamo Nico."}]}, config=config)
r = agente.invoke({"messages": [{"role": "user", "content": "Come mi chiamo?"}]}, config=config)
print(r["messages"][-1].text)   # ora ricorda: stesso thread_id

### Output atteso

Una risposta che ricorda il nome `Nico`.

### Spiegazione del blocco · Thread diverso, isolamento

Cambiare identificatore crea una conversazione separata. Questo confine evita che informazioni di utenti o sessioni diverse si mescolino.

In [ ]:
# Cambiando thread_id ripartiamo da zero: memoria isolata per conversazione.
altro = {"configurable": {"thread_id": "conversazione-2"}}
r2 = agente.invoke({"messages": [{"role": "user", "content": "Come mi chiamo?"}]}, config=altro)
print(r2["messages"][-1].text)   # non lo sa: è un'altra conversazione

### Output atteso

Il modello non conosce il nome nella nuova conversazione.

## 3 · Memoria di lungo termine con uno store + tool

La memoria di breve termine vive dentro una conversazione. Per ricordare *tra* conversazioni
serve uno **store** persistente. Lo simuliamo con un dizionario e due tool per salvare/leggere.

### Spiegazione del blocco · Store di lungo termine

Un dizionario simula memoria esterna al thread. I tool rendono esplicita la decisione di salvare o recuperare, anziché memorizzare automaticamente tutto.

In [ ]:
# Uno store globale semplicissimo (un dizionario). In produzione: un DB o LangGraph Store.
from langchain_core.tools import tool

STORE: dict[str, str] = {}


@tool
def ricorda(chiave: str, valore: str) -> str:
    """Salva un'informazione durevole da ricordare in futuro."""
    STORE[chiave] = valore
    return f"Memorizzato '{chiave}'."


@tool
def richiama(chiave: str) -> str:
    """Recupera un'informazione salvata in precedenza."""
    return STORE.get(chiave, "(niente in memoria per questa chiave)")

### Output atteso

Nessun output. `STORE` è vuoto e i due tool sono disponibili.

### Spiegazione del blocco · Scrittura della preferenza

L'agente riceve istruzione esplicita di usare `ricorda`. Il dato sopravvive alla creazione di un nuovo agente perché vive nello store esterno.

In [ ]:
print("Store prima del salvataggio:", STORE)
# Prima conversazione: l'agente salva una preferenza.
agente_memoria = create_agent(model=model, tools=[ricorda, richiama])
agente_memoria.invoke({"messages": [{
    "role": "user",
    "content": "Ricorda che preferisco risposte in elenco puntato. Usa il tool ricorda.",
}]})
print("Store dopo il salvataggio:", STORE)

### Output atteso

`Store dopo il salvataggio:` seguito da una coppia chiave/valore relativa alla preferenza.

### Spiegazione del blocco · Recupero tra conversazioni

Un nuovo agente non ha la cronologia precedente, ma può interrogare lo store. È la differenza tra memoria conversazionale e memoria durevole.

In [ ]:
# Seconda conversazione (nuovo agente, nessuna memoria di breve termine):
# eppure recupera la preferenza dallo store di lungo termine.
agente_nuovo = create_agent(model=model, tools=[ricorda, richiama])
r = agente_nuovo.invoke({"messages": [{
    "role": "user",
    "content": "Come preferisco le risposte?",
}]})
print(r["messages"][-1].text)

print("TUTTI I MESSAGGI:", r["messages"])

### Output atteso

Una risposta che riferisce la preferenza per elenchi puntati.

### Spiegazione del blocco · Scoperta e trace della memoria

Il tool `elenca_memoria` permette al nuovo agente di scoprire quali chiavi sono disponibili prima di recuperarne una. Il trace rende osservabili richiesta del tool, risultato e risposta finale.

In [ ]:
@tool
def elenca_memoria() -> str:
    """Elenca tutte le chiavi presenti in memoria."""
    if not STORE:
        return "(memoria vuota)"
    return "Chiavi: " + ", ".join(STORE.keys())

print("Memoria:", elenca_memoria.invoke({}))

agente_nuovo = create_agent(model=model, tools=[ricorda, richiama, elenca_memoria])
r = agente_nuovo.invoke({"messages": [
    {
    "role":"system",
    "content": "Quando hai bisogno di salvare una memoria, usa il tool `ricorda`. Quando vuoi ricordare qualcosa, usa i tool `elenca_memoria` e `richiama`."
    },
    {
    "role": "user",
    "content": "Come preferisco le risposte?",
}]})

# Stampiamo la sequenza di messaggi per vedere il ragionamento in azione.
for i, m in enumerate(r["messages"]):
    tipo = type(m).__name__
    if getattr(m, "tool_calls", None):        # il modello ha CHIESTO di usare un tool
        print(f"{i}. {tipo}: chiama {m.tool_calls[0]['name']} con {m.tool_calls[0]['args']}")
    elif tipo == "ToolMessage":               # il RISULTATO del tool
        print(f"{i}. {tipo}: risultato = {m.content}")
    else:
        print(f"{i}. {tipo}: {str(m.content)[:80]}")


### Output atteso

La chiave salvata compare nell'elenco; il trace mostra le chiamate a `elenca_memoria` e `richiama`, quindi una risposta che recupera la preferenza.

## Prova tu

- Aggiungi al system prompt: "All'inizio controlla sempre la memoria con `richiama`."
- Con conversazioni lunghe, il contesto cresce: cerca `SummarizationMiddleware` per riassumerlo.

**Idea chiave**: *breve termine* = stato della conversazione (checkpoint + thread_id);
*lungo termine* = uno store esterno che l'agente consulta con dei tool.

## Laboratorio aggiuntivo

Gli esempi seguenti riusano quanto costruito sopra. Il primo amplia il caso normale; il
secondo esercita un confine, un errore o una proprietà che spesso causa bug reali.

## Esempio aggiuntivo: aggiornamento e chiave assente

### Spiegazione del blocco

Una memoria utile deve definire comportamento per sovrascrittura e mancata corrispondenza.

In [ ]:
print(ricorda.invoke({"chiave": "formato", "valore": "tabella"}))
print(ricorda.invoke({"chiave": "formato", "valore": "elenco"}))
print("formato:", richiama.invoke({"chiave": "formato"}))
print("lingua:", richiama.invoke({"chiave": "lingua"}))

### Output atteso

La seconda scrittura sostituisce la prima; `formato` vale `elenco`, mentre `lingua` restituisce il fallback.

## Esempio aggiuntivo: ispezionare il checkpoint

### Spiegazione del blocco

Dopo le invocazioni è possibile leggere lo stato senza chiedere di nuovo al modello.

In [ ]:
stato = agente.get_state(config)
messaggi_salvati = stato.values.get("messages", [])
print("Messaggi nel thread 1:", len(messaggi_salvati))
print("Ultimo tipo:", type(messaggi_salvati[-1]).__name__)

### Output atteso

Numero maggiore di zero e ultimo tipo normalmente `AIMessage`. Nessuna chiamata API aggiuntiva.

## Riepilogo e troubleshooting

Prima di proseguire, prova a spiegare con parole tue: quale stato è cambiato, quale
componente ha preso la decisione e quale prova rende osservabile l'esito.

Se una cella fallisce:

1. rileggi l'output atteso e individua la prima invariante non rispettata;
2. verifica di aver eseguito tutte le celle precedenti nello stesso kernel;
3. per i notebook live, controlla `.env`, modello disponibile e quota API;
4. riavvia il kernel solo dopo aver conservato eventuali file che vuoi ispezionare;
5. non correggere un caso negativo: l'errore previsto è parte dell'esempio.